## 07. nanobind: bind a hand-written C++ kernel

One way to get more performance is to write the kernel in C++ and bind it back to Python. This notebook takes the binding-library and FFI (foreign function interface) approach with **nanobind**, the modern successor to pybind11.

### Table of Contents

1. [Imports and source layout](#sec1)
2. [The C++ source](#sec2)
3. [Build via nanobind + CMake](#sec3)
4. [Acceptance and timing](#sec4)
5. [nanobind vs pybind11](#sec5)
6. [Limitation: binding and build glue](#sec6)

### <a id="sec1"></a>1. Imports and source layout

The C++ source lives at `notebooks/swe_step.cpp`. We invoke `cmake` and
the system C++ compiler via `subprocess`. The resulting shared library is
imported back into Python.

---

**Quick Docs**

- `nanobind.cmake_dir()`: path to nanobind's CMake config files. We pass
  it via `-Dnanobind_DIR=` so `find_package(nanobind CONFIG REQUIRED)`
  succeeds.
- `nanobind_add_module(target source)`: nanobind's CMake helper that
  creates a Python extension target with the right include paths,
  link flags, and visibility settings.
- `nb::ndarray<T, nb::ndim<1>>` (on the C++ side): a typed NumPy view (`double*` + `shape`).
  No copy unless the layout demands one.

In [ ]:
import sys, subprocess, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import nanobind

import swe_core

# Shared problem parameters.
N       = 16384
L       = 10.0
H0      = 1.0
AMP     = 0.1
SIG     = 0.5
CFL     = 0.4
G       = 9.81
N_STEPS = 1000
dx      = L / N
DT      = swe_core.fixed_dt(H0 + AMP, dx, cfl=CFL, g=G)

# CMakeLists.txt + swe_step.cpp live next to swe_core.py; build artifacts go in build/.
CPP_DIR   = Path(swe_core.SWE_STEP_CPP).parent
BUILD_DIR = CPP_DIR / 'build'

# Reference NumPy field for the acceptance gate.
def run_numpy_reference():
    h, hu = swe_core.bump_ic(N, L=L, h0=H0, amplitude=AMP, sigma=SIG)
    for _ in range(N_STEPS):
        swe_core.apply_bc_reflective(h, hu)
        h, hu = swe_core.step_numpy(h, hu, dx, DT, g=G)
    return h, hu

h_ref, hu_ref = run_numpy_reference()

### <a id="sec2"></a>2. The C++ source

Same **Read / Compute / Update** shape as NB 05, expressed in C++ via nanobind:

- **Read**. Scalar `double` loads from the `double*` pointer nanobind hands us, a zero-copy view of the NumPy buffer.
- **Compute**. An inline `rusanov_face()` helper does the scalar arithmetic, all in registers.
- **Update**. Scalar writes through `double*`. No Python objects on the hot path.

From `swe_step.cpp`:

```cpp
#include <nanobind/nanobind.h>
#include <nanobind/ndarray.h>
#include <cmath>
#include <algorithm>

namespace nb = nanobind;

// Scalar face flux; small body so g++ -O3 can vectorise the outer loop.
inline void rusanov_face(double hL, double hR, double huL, double huR,
                         double g, double& Fh, double& Fhu) {
    /* ...same arithmetic as swe_core.step_numpy... */
}

void cpp_step(
    nb::ndarray<const double, nb::ndim<1>> h_in,
    nb::ndarray<const double, nb::ndim<1>> hu_in,
    nb::ndarray<double, nb::ndim<1>> h_out,
    nb::ndarray<double, nb::ndim<1>> hu_out,
    double dx, double dt, double g)
{
    const double* h  = h_in.data();
    /* ...one forward-Euler Rusanov step over interior cells... */
}

NB_MODULE(swe_step, m) {
    m.def("cpp_step", &cpp_step,
          nb::arg("h"), nb::arg("hu"), nb::arg("h_new"), nb::arg("hu_new"),
          nb::arg("dx"), nb::arg("dt"), nb::arg("g") = 9.81);
}
```

Three things to note: `nb::ndarray<const double, nb::ndim<1>>` is a zero-copy typed view into the NumPy buffer, `NB_MODULE`'s name must match the compiled library filename (`swe_step.cpython-...so`), and `nb::arg` carries Python keyword names and defaults into the generated signature.

### <a id="sec3"></a>3. Build via nanobind + CMake

nanobind ships with a CMake helper, `nanobind_add_module(...)`, that
sets up include paths, link flags, RPATH, and visibility for you. Let's
write a 6-line `CMakeLists.txt` and then configure and build the library.

---

**Quick Docs**

- `cmake -B build -S .` configures (out-of-source build directory).
- `-DCMAKE_BUILD_TYPE=Release` is essential for the compiler to emit optimisation flags (`-O3 -DNDEBUG`).
- `-DPython_EXECUTABLE=` ensures we correctly link against the Python executable in our venv.
- `cmake --build build --config Release -j` compiles our code into a shared library.

In [ ]:
NANOBIND_CMAKE_DIR = nanobind.cmake_dir()

CMAKELISTS = '''
cmake_minimum_required(VERSION 3.20)
project(swe_step LANGUAGES CXX)
set(CMAKE_CXX_STANDARD 17)
find_package(Python 3.12 REQUIRED COMPONENTS Interpreter Development.Module)
find_package(nanobind CONFIG REQUIRED)
nanobind_add_module(swe_step swe_step.cpp)
'''
(CPP_DIR / 'CMakeLists.txt').write_text(CMAKELISTS)

def run(cmd, **kw):
    r = subprocess.run(cmd, cwd=str(CPP_DIR), capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f'CMD FAILED: {cmd}\n--- stderr ---\n{r.stderr}')
    return r

# TODO: configure + build the C++ extension.
#   - cmake -B build -S . with these flags:
#       -DCMAKE_BUILD_TYPE=Release           (so g++ gets -O3)
#       -DPython_EXECUTABLE=<sys.executable> (the venv interpreter)
#       -Dnanobind_DIR=NANOBIND_CMAKE_DIR    (so find_package(nanobind) works)
#   - cmake --build build --config Release -j
# Hint: use the `run([...])` helper above.
# TODO: cmake configure step
...
# TODO: cmake build step
...

Now that we have our shared library, let's try and import it on the Python side:

In [ ]:
# Import the freshly-built module. The .so lives in build/.
sys.path.insert(0, str(BUILD_DIR))
import swe_step as cpp_swe

### <a id="sec4"></a>4. Acceptance and timing

The time loop reuses two pre-allocated buffers, swapping input and output each
step (double buffering), so the timing measures the kernel call, not per-step
allocation. Boundary conditions are re-applied on the Python side
(`swe_core.apply_bc_reflective`) between steps, so the C++ kernel only does the
work we're benchmarking.

In [ ]:
def run_nanobind() -> tuple[np.ndarray, np.ndarray]:
    h, hu = swe_core.bump_ic(N, L=L, h0=H0, amplitude=AMP, sigma=SIG)
    h2 = np.empty_like(h); hu2 = np.empty_like(hu)
    for _ in range(N_STEPS):
        swe_core.apply_bc_reflective(h, hu)
        cpp_swe.cpp_step(h, hu, h2, hu2, dx, DT, G)
        h, hu, h2, hu2 = h2, hu2, h, hu
    return h, hu

# Cold = first call after import
t0 = time.perf_counter()
h_nb, _ = run_nanobind()
cold_s = time.perf_counter() - t0
print(f'cold  N={N} x {N_STEPS} steps: {cold_s*1000:7.1f} ms  (incl. shared-library dlopen)')

warm = swe_core.timed_run(run_nanobind, warmup=2, repeats=5, label='05_nanobind')
print(f'warm  N={N} x {N_STEPS} steps: median {warm["median_s"]*1000:7.1f} ms')

diff = swe_core.max_diff(h_ref, h_nb)
print(f'max_diff(nanobind_fp64, numpy_fp64) = {diff:.3e}')
TOL = 1e-12
assert diff < TOL, f'FAIL: {diff:.3e} >= {TOL:.0e}'
print(f'PASS: within {TOL:.0e} of the NumPy reference.')

swe_core.save_timing(
    warm, grid_str=f'N={N}', tool='nanobind', hardware='cpu',
    dtype='float64', steps=N_STEPS,
    cold_s=cold_s, max_diff_vs_numpy=diff, binding='nanobind',
    build_type='Release',
)

### <a id="sec5"></a>5. nanobind vs pybind11

pybind11 is nanobind's predecessor and remains the most widely used C++↔Python binding library (PyTorch, SciPy, and much of the HPC ecosystem). The ergonomics are nearly identical: in pybind11 the array type is `py::array_t<double, py::array::c_style>` and the entry point is `PYBIND11_MODULE`. nanobind's published benchmarks report up to ~10× lower per-call overhead, ~4× faster compilation, and ~5× smaller binaries ([nanobind](https://github.com/wjakob/nanobind)).

### <a id="sec6"></a>6. Limitation: binding and build glue

This notebook's cost is everything around the kernel, not the kernel itself:

- **The bindings track the C++ API.** `NB_MODULE`, `m.def`, and `nb::arg` are a separate layer that has to be hand-edited whenever the C++ source evolves: a new function or changed signature requires updating the bindings to match. The kernel is also no longer regular C++, it carries nanobind types like `nb::ndarray`. CppJIT (NB 05) needs neither, compiling standard C++/CUDA with no bindings library on the C++ side.
- **Build system.** Building the module requires working with CMake and a C++ toolchain.
- **Recompile-on-edit cycle.** Editing the source leads to recompilation, restarting the kernel and re-importing the module.
- **FFI boundary cost.** Each call from Python into C++ pays a small fixed overhead for argument conversion. At ~1000 calls per run it stays well under a millisecond, negligible against the kernel time.

The warm time above lands close to the NumPy baseline at this size, but runs slower, and does not scale. The single-threaded scalar loop has little to exploit when the arrays are cache-resident while NumPy's element-wise ops are SIMD-vectorised. This is not a shortcoming of nanobind, but rather our kernel implementation. To outperform NumPy, we require multi-threading (as in NB 03) or device parallelism (GPU).

---

**Recap.**

- A short `CMakeLists.txt` + `nb::ndarray<...>` exposes a C++ kernel to Python with zero-copy NumPy arrays.
- The cost is the glue around the kernel: a bindings layer to maintain as the C++ evolves, plus build and FFI overhead.
- At this size the scalar kernel is on par/slower than NumPy, but does not scale.

Next: `08__swe__cppjit__thrust.ipynb` uses on-the-fly JIT-compiled C++, with no CMake, recompilation, or separate shared library. It provides automatic Python bindings and runs the whole solve on the GPU using the thrust library.